<a href="https://colab.research.google.com/github/EduardoSilvaVergara/Act2_4_1/blob/main/Act2_4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload()

Saving ventas_sucias.csv to ventas_sucias.csv


{'ventas_sucias.csv': b'id,nombre,edad,ciudad,fecha_compra,monto,metodo_pago\n1,Juan Perez,25,Santiago,2023-01-10,10000,tarjeta\n2,Maria Lopez, ,Valparaiso,10/02/2023,15000,efectivo\n3,Carlos Ruiz,30,santiago,2023/03/15,20000,Transferencia\n4,Ana Torres,28,Vi\xc3\xb1a del Mar,2023-04-20, ,tarjeta\n5,Pedro Gomez,45,VALPARAISO,2023-05-05,30000,efectivo\n5,Pedro Gomez,45,VALPARAISO,2023-05-05,30000,efectivo\n6,Luis Diaz,150,Santiago,2023-06-01,5000,efectivo\n7,Sofia Rojas,22,Vi\xc3\xb1a Del Mar,2023-07-12,12000,debito\n8, ,35,Santiago,2023-08-10,18000,tarjeta\n9,Diego Soto,27,Valparaiso,2023-13-01,22000,transferencia\n'}

In [15]:
!pip install pandas psycopg2-binary

In [25]:
import pandas as pd

df = pd.read_csv("ventas_sucias.csv")

# =========================
# NORMALIZACIÓN
# =========================
df['nombre'] = df['nombre'].astype(str).str.strip()
df['ciudad'] = df['ciudad'].astype(str).str.strip().str.lower()
df['metodo_pago'] = df['metodo_pago'].astype(str).str.strip().str.lower()

# =========================
# CONVERSIÓN TIPOS (estructural)
# =========================
df['edad'] = pd.to_numeric(df['edad'], errors='coerce')
df['monto'] = pd.to_numeric(df['monto'], errors='coerce')
df['fecha_compra'] = pd.to_datetime(df['fecha_compra'], errors='coerce')

# =========================
# VALIDACIONES ESTRUCTURALES
# =========================
df['val_nombre'] = df['nombre'].notna() & (df['nombre'] != "")
df['val_edad_tipo'] = df['edad'].notna()
df['val_monto_tipo'] = df['monto'].notna()
df['val_fecha'] = df['fecha_compra'].notna()
df['val_id_unico'] = ~df['id'].duplicated()

# =========================
# VALIDACIONES SEMÁNTICAS
# =========================
metodos_validos = ['tarjeta', 'efectivo', 'transferencia', 'debito']
ciudades_validas = ['santiago', 'valparaiso', 'viña del mar']

df['val_edad_rango'] = df['edad'].between(0,120)
df['val_monto_positivo'] = df['monto'] > 0
df['val_metodo_pago'] = df['metodo_pago'].isin(metodos_validos)
df['val_ciudad'] = df['ciudad'].isin(ciudades_validas)
df['val_fecha_logica'] = df['fecha_compra'] < pd.Timestamp.today()

# =========================
# VALIDACIÓN FINAL
# =========================
df['valido'] = (
    df['val_nombre'] &
    df['val_edad_tipo'] &
    df['val_monto_tipo'] &
    df['val_fecha'] &
    df['val_id_unico'] &
    df['val_edad_rango'] &
    df['val_monto_positivo'] &
    df['val_metodo_pago'] &
    df['val_ciudad'] &
    df['val_fecha_logica']
)

# =========================
# SEPARACIÓN
# =========================
validos = df[df['valido'] == True]
rechazados = df[df['valido'] == False]

# Guardar evidencia
validos.to_csv("validated.csv", index=False)
rechazados.to_csv("rechazados.csv", index=False)

In [26]:
import psycopg2

conn = psycopg2.connect(
    host="ep-floral-lab-ac48n63j.sa-east-1.aws.neon.tech",
    database="neondb",
    user="neondb_owner",
    password="npg_4d1OGlTyVujk",
    port="5432"
)

cursor = conn.cursor()

print("Conectado!")

Conectado!


In [27]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS ciudad (
    id SERIAL PRIMARY KEY,
    nombre VARCHAR(50) UNIQUE NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS ventas (
    id INT PRIMARY KEY,
    nombre VARCHAR(100),
    edad INT,
    ciudad_id INT,
    fecha_compra DATE,
    monto INT,
    metodo_pago VARCHAR(50),
    FOREIGN KEY (ciudad_id) REFERENCES ciudad(id)
);
""")

conn.commit()

In [28]:
ciudades = validos['ciudad'].dropna().unique()

for ciudad in ciudades:
    cursor.execute(
        "INSERT INTO ciudad (nombre) VALUES (%s) ON CONFLICT DO NOTHING",
        (ciudad,)
    )

conn.commit()

In [29]:
# =========================
# 1. ASEGURAR CREACIÓN DEL LOG
# =========================
with open("carga.log", "w") as f:
    f.write("Inicio del proceso de carga\n")

# =========================
# 2. CONFIGURAR LOGGING
# =========================
import logging

logging.basicConfig(
    filename="carga.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True  # 🔥 importante en Colab
)

logging.info("=== INICIO CARGA DE DATOS ===")

# =========================
# 3. CONTADORES
# =========================
insertados = 0
ignorados = 0
errores = 0

# =========================
# 4. PROCESO DE CARGA
# =========================
for _, row in validos.iterrows():
    try:
        # Obtener ciudad_id
        cursor.execute("SELECT id FROM ciudad WHERE nombre=%s", (row['ciudad'],))
        result = cursor.fetchone()

        if result is None:
            raise Exception("Ciudad no encontrada")

        ciudad_id = result[0]

        # Insert con control de duplicados
        cursor.execute("""
        INSERT INTO ventas (id, nombre, edad, ciudad_id, fecha_compra, monto, metodo_pago)
        VALUES (%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT (id) DO NOTHING
        """, (
            int(row['id']),
            row['nombre'],
            int(row['edad']),
            ciudad_id,
            row['fecha_compra'],
            int(row['monto']),
            row['metodo_pago']
        ))

        # Verificar si insertó o no
        if cursor.rowcount == 0:
            ignorados += 1
            logging.warning(f"ID {row['id']} ya existía, no insertado")
        else:
            insertados += 1
            logging.info(f"Insertado ID {row['id']} correctamente")

    except Exception as e:
        errores += 1
        conn.rollback()  # 🔥 evita bloqueo de transacción
        logging.error(f"Error en fila {row['id']}: {e}")

# =========================
# 5. COMMIT FINAL
# =========================
conn.commit()

# =========================
# 6. RESUMEN FINAL
# =========================
logging.info("=== RESUMEN ===")
logging.info(f"Insertados: {insertados}")
logging.info(f"Ignorados (duplicados): {ignorados}")
logging.info(f"Errores: {errores}")
logging.info("=== FIN DEL PROCESO ===")

# 🔥 cerrar log correctamente
logging.shutdown()

In [30]:
import os
print(os.listdir())

['.config', 'ventas_sucias.csv', 'carga.log', 'validated.csv', 'rechazados.csv', 'sample_data']


In [33]:
files.download("validated.csv")
files.download("rechazados.csv")
files.download("carga.log")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
cursor.execute("DROP TABLE IF EXISTS ventas;")
cursor.execute("DROP TABLE IF EXISTS ciudad;")
conn.commit()